# Multi-Domain Retrieval Pipeline

迭代式多域蛋白质生成：
1. DOMIN 检索与 query_domain 匹配的下一个 domain
2. DOMO 基于原始 domain 列表合成序列

In [2]:
%load_ext autoreload
%autoreload 2

ROOT_DIR = "/sujin/PycharmProjects/DOMINO"

import sys
sys.path.append(ROOT_DIR)

import torch
import faiss
import numpy as np
from tqdm import tqdm
import yaml

In [1]:
1

1

## 1. 模型初始化

In [4]:
import sys
sys.path.append(str(ROOT_DIR / "src"))
sys.path.append(f"{ROOT_DIR}/src/DOMO")
sys.path.append(str(ROOT_DIR))

import torch
import faiss
import numpy as np
from tqdm import tqdm
import yaml

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [3]:
# 初始化 DOMIN 模型
from src.DOMIN.models.ted.ted_domain_model import TedDomainModel

config_path = f"{ROOT_DIR}/weights/DOMIN"
domin_checkpoint = f"{ROOT_DIR}/weights/DOMIN/DOMIN.pt"
model_config = {
    "config_path": config_path,
    "from_checkpoint": domin_checkpoint,
}

domin_model = TedDomainModel(**model_config)
domin_model.to(device).eval()
print(f"DOMIN model loaded. Temperature: {domin_model.model.temperature.item():.4f}")

/root/miniconda3/envs/DOMINO/lib/python3.11/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/root/miniconda3/envs/DOMINO/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/DOMINO/lib/python3.11/site-packages/transformers/loss/loss_for_object_detection.py:28: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.26.0)
  from scipy.optimize import linear_sum_assignment


[2026-05-14 10:05:02,646] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/root/miniconda3/envs/DOMINO/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH
 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2.0 but detected 2.6
 [WARNING]  using untested triton version (3.2.0), only 1.0.0 is known to be compatible


/root/miniconda3/envs/DOMINO/lib/python3.11/site-packages/deepspeed/runtime/zero/linear.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @autocast_custom_fwd
/root/miniconda3/envs/DOMINO/lib/python3.11/site-packages/deepspeed/runtime/zero/linear.py:66: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @autocast_custom_bwd


No lr_scheduler_kwargs provided. The default learning rate is 0.
No optimizer_kwargs provided. The default optimizer is AdamW.
Some weights of the model checkpoint were not used: ['esm.embeddings.position_ids']


NameError: name 'device' is not defined

In [4]:
# 初始化 DOMO 模型
from src.DOMO.utils.init_utils import construct_class_by_name

domo_config_path = ROOT_DIR / "src" / "DOMO" / "configs" / "DOMO_config.yaml"
with open(domo_config_path, 'r') as f:
    domo_config = yaml.safe_load(f)

domo_checkpoint = ROOT_DIR / "weights" / "DOMO" / "pytorch_model.bin"

domo_model = construct_class_by_name(**domo_config["Model"]["kwargs"], logger=print)
domo_model.load_state_dict(torch.load(domo_checkpoint, map_location=device))
domo_model.eval()
domo_model = domo_model.to(device)

tokenizer = domo_model.tokenizer
print("DOMO model loaded successfully")

ModuleNotFoundError: No module named 'src'

In [ ]:
# 加载 FAISS 索引和序列映射
INDEX_DIR = Path("/sujin/Datasets/TED/embedding/afdb_cluster_power0.75/")
INDEX_PATH = INDEX_DIR / "key_embeddings.index"
TSV_PATH = INDEX_DIR / "sampled_proteins.tsv"

faiss_index = faiss.read_index(str(INDEX_PATH), faiss.IO_FLAG_MMAP)
print(f"FAISS index loaded: {faiss_index.ntotal:,} vectors, dim={faiss_index.d}")

print(f"Loading sequence mapping from: {TSV_PATH}")

idx_to_segment = []
with open(TSV_PATH, 'r') as f:
    for line in tqdm(f, desc="Loading segments"):
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            idx_to_segment.append(parts[1])
        else:
            idx_to_segment.append(parts[0])

print(f"Loaded {len(idx_to_segment):,} segments")
print(f"Example segment (idx=0): {idx_to_segment[0][:80]}...")

## 2. 定义 Pipeline 函数

In [ ]:
def retrieve_next_domain(query_domain: str, top_k: int = 1, nprobe: int = 1) -> list:
    """
    使用 DOMIN 检索与 query_domain 匹配的下一个 domain
    
    Args:
        query_domain: 查询域的序列
        top_k: 返回前 k 个候选
        nprobe: FAISS 搜索时探查的聚类中心数
    
    Returns:
        List of (index, distance, segment) tuples
    """
    faiss_index.nprobe = nprobe
    
    with torch.no_grad():
        query_repr = domin_model.get_query_repr(query_domain).cpu().numpy()
    
    distances, indices = faiss_index.search(query_repr.reshape(1, -1).astype('float32'), top_k + 1)
    
    results = []
    for idx, dist in zip(indices[0].tolist(), distances[0].tolist()):
        if idx != -1 and dist != -1:
            segment = idx_to_segment[idx] if idx < len(idx_to_segment) else None
            results.append((idx, dist, segment))
    
    return results

print("retrieve_next_domain() function defined")

In [ ]:
def generate_combined_sequence(domain_list: list) -> str:
    """
    使用 DOMO 将多个 domain 合成一个序列
    
    Args:
        domain_list: domain 序列列表
    
    Returns:
        合成的蛋白质序列
    """
    tokenized_domain = tokenizer(domain_list, return_tensors="pt", padding=True, truncation=True, max_length=512)
    domain_ids = tokenized_domain.input_ids.to(device)
    domain_masks = tokenized_domain.attention_mask.to(device)
    num_domains_per_protein = torch.tensor([len(domain_list)]).to(device)
    
    with torch.no_grad():
        result = domo_model.generate(
            domain_ids=domain_ids,
            domain_masks=domain_masks,
            num_domains_per_protein=num_domains_per_protein
        )
    
    return result["output_seqs"][0]

print("generate_combined_sequence() function defined")

## 3. 示例：迭代生成多域蛋白质

In [ ]:
# 示例初始 query domain
example_query_domain = "TcGvDvRvKdRdEdFdLfEdLwGdRcKdAqGvRdFpPpAwAiSdTgSpNvGgEiIfSgIeWf<unk>ElEsRnRvRqPlAlEvNqAlRvLlTlHvGvLlLcRvEvRlDlIfPdVfLsSdDsRnShPsIwVtPwVgLwVqGqEaDdRvMlClKvRqMlSqAvLqPcLcEvRpHvGsAyYhVwQdAwIdDdApPpSnVdPdArGrErEiItLgRtIgArPrShAsVsHdEdTpEvEnIsHvRvFsVsDvAsLsDsGvIsWcSvEvLsGv"

domain_pool = [example_query_domain]

print(f"Initial query domain: {example_query_domain[:50]}...")
print(f"Domain length (AA): {len(example_query_domain.replace('<unk>', '')) // 2}")

In [ ]:
# === 迭代 1: 检索第一个 target domain ===
print("\n" + "="*60)
print("迭代 1: DOMIN 检索")
print("="*60)

retrieved_1 = retrieve_next_domain(example_query_domain, top_k=5, nprobe=10)
print(f"Top 5 retrieved domains:")
for i, (idx, dist, seg) in enumerate(retrieved_1):
    seg_preview = seg[:40] + "..." if seg and len(seg) > 40 else seg
    print(f"  {i+1}. idx={idx}, dist={dist:.4f}, seg={seg_preview}")

target_idx_1, target_dist_1, target_seg_1 = retrieved_1[0]
domain_pool.append(target_seg_1)

print(f"\nSelected: idx={target_idx_1}, distance={target_dist_1:.4f}")

In [ ]:
# === DOMO 合成 2-domain 蛋白质 ===
print("\n" + "="*60)
print("DOMO 合成 2-domain 蛋白质")
print("="*60)

print(f"Domain pool for synthesis: {len(domain_pool)} domains")
for i, d in enumerate(domain_pool):
    print(f"  Domain {i+1}: {d[:40]}...")

sequence_2dom = generate_combined_sequence(domain_pool)
print(f"\n合成结果 (2-domain):")
print(f"Sequence: {sequence_2dom}")
print(f"Length: {len(sequence_2dom)} AA")

In [ ]:
# === 迭代 2: 用合成的 2-domain 序列检索第三个 domain ===
print("\n" + "="*60)
print("迭代 2: DOMIN 检索")
print("="*60)

query_for_iter2 = sequence_2dom
print(f"Query (synthesized 2-domain): {query_for_iter2[:50]}...")

retrieved_2 = retrieve_next_domain(query_for_iter2, top_k=5, nprobe=10)
print(f"\nTop 5 retrieved domains:")
for i, (idx, dist, seg) in enumerate(retrieved_2):
    seg_preview = seg[:40] + "..." if seg and len(seg) > 40 else seg
    print(f"  {i+1}. idx={idx}, dist={dist:.4f}, seg={seg_preview}")

target_idx_2, target_dist_2, target_seg_2 = retrieved_2[0]

if target_seg_2 not in domain_pool:
    domain_pool.append(target_seg_2)
    print(f"\nSelected new domain: idx={target_idx_2}, distance={target_dist_2:.4f}")
else:
    print(f"\nTop 1 is already in pool, checking next...")
    for idx, dist, seg in retrieved_2[1:]:
        if seg not in domain_pool:
            domain_pool.append(seg)
            print(f"Selected: idx={idx}, distance={dist:.4f}")
            break

print(f"\nDomain pool now has {len(domain_pool)} domains")

In [ ]:
# === DOMO 合成 3-domain 蛋白质 ===
print("\n" + "="*60)
print("DOMO 合成 3-domain 蛋白质")
print("="*60)

print(f"Domain pool for synthesis: {len(domain_pool)} domains")
for i, d in enumerate(domain_pool):
    print(f"  Domain {i+1}: {d[:40]}...")

sequence_3dom = generate_combined_sequence(domain_pool)
print(f"\n合成结果 (3-domain):")
print(f"Sequence: {sequence_3dom}")
print(f"Length: {len(sequence_3dom)} AA")

## 4. 完整 Pipeline 函数

In [ ]:
def iterative_domain_synthesis(
    initial_domain: str,
    num_iterations: int = 3,
    nprobe: int = 10,
    verbose: bool = True
) -> dict:
    """
    迭代式多域蛋白质合成
    
    Args:
        initial_domain: 初始 domain 序列
        num_iterations: 迭代次数
        nprobe: FAISS 检索参数
        verbose: 是否打印详细信息
    
    Returns:
        dict: 包含 'domain_pool', 'final_sequence', 'iterations' 等信息
    """
    domain_pool = [initial_domain]
    synthesis_history = []
    
    for i in range(1, num_iterations):
        if verbose:
            print(f"\n{'='*60}")
            print(f"迭代 {i}: DOMIN 检索")
            print(f"{'='*60}")
        
        if i == 1:
            query = initial_domain
        else:
            query = synthesis_history[-1]['sequence']
        
        retrieved = retrieve_next_domain(query, top_k=5, nprobe=nprobe)
        
        selected = None
        for idx, dist, seg in retrieved:
            if seg not in domain_pool:
                selected = {'idx': idx, 'dist': dist, 'seg': seg}
                break
        
        if selected is None:
            if verbose:
                print("Warning: 所有检索到的 domain 已在 pool 中")
            break
        
        domain_pool.append(selected['seg'])
        
        if verbose:
            print(f"Selected domain idx={selected['idx']}, dist={selected['dist']:.4f}")
            print(f"Domain pool size: {len(domain_pool)}")
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"DOMO 合成 {len(domain_pool)}-domain 蛋白质")
            print(f"{'='*60}")
        
        sequence = generate_combined_sequence(domain_pool)
        
        synthesis_history.append({
            'iteration': i,
            'domain_pool': domain_pool.copy(),
            'selected_domain': selected,
            'sequence': sequence
        })
        
        if verbose:
            print(f"Synthesized sequence: {sequence[:50]}...")
            print(f"Length: {len(sequence)} AA")
    
    return {
        'domain_pool': domain_pool,
        'final_sequence': synthesis_history[-1]['sequence'] if synthesis_history else initial_domain,
        'iterations': synthesis_history
    }

print("iterative_domain_synthesis() function defined")